# PREP-02 Retrieve, cite, or stop (Day 2 words)

**Time:** 50–70 minutes &nbsp;|&nbsp; **Maps to:** T1-06 … T1-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/prep/PREP-02_retrieve_and_cite.ipynb)

Or: **File → Upload notebook** at [colab.research.google.com](https://colab.research.google.com/) and choose `PREP-02_retrieve_and_cite.ipynb`.


**How to use this notebook (no coding)**

1. **Runtime → Change runtime type → CPU** (not GPU). You do not need a graphics card.
2. Click ▶ on each cell from the top, or **Runtime → Run all**.
3. Read the note *under* a cell after it runs.
4. Change **only** lines marked `# [CHANGE]` — usually one word inside quotes `"..."`. Do not delete the quotes.
5. Nothing from work goes in a cell. CivicDesk names are fiction.

You are not learning Python. You are practising the same click-and-read path that no-code people use on Day 1.


## Why this night exists

Day 1 the model had **no handbook** in the prompt, so it could invent Circular 99. Day 2 fixes that without training:

**RAG** = **R**etrieve an *approved* paragraph, **A**ugment the prompt with it, then **G**enerate. If nothing approved is found, **abstain** (say you cannot cite, and stop).

```
  question
     │
     ▼
  search a small approved pile   (not the whole ministry)
     │
     ▼
  access gate   (this role may / may not see that paragraph)
     │
     ▼
  cite a quoted span     or     I cannot cite. Stop.
```

Optional reading: [IBM on RAG](https://www.ibm.com/think/topics/retrieval-augmented-generation) (plain language). Skip the vendor diagrams.

Paper that later becomes a PDF is **not** cleaned in this notebook. If your ministry scans files, skim [Tesseract: improving output quality](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html) (resolution, skew, image-only PDF). Do not install Tesseract. Do not upload a real page. Technical staff who already know tables may glance at [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html); Day 2 is INDEX vs NEVER, not a pandas lab.


## 1. Five labels (you will argue these on Day 2 morning)

Many EOIs say “we need an LLM.” Often they need one of these instead.

| Label | You change… | CivicDesk example |
|---|---|---|
| **prompt** | The wall instruction | “Never hire. Never reset a password.” |
| **RAG** | Which approved paragraph is copied in this turn | Overnight attachment circular |
| **integration** | A query to a **register** or ticket store | Which printers are out of warranty |
| **fine-tune** | A small extra file on frozen weights (Day 3: inspect, do not train) | Stable phrasing of C2 |
| **none** | Leave it with a person or a non-LM tool | Patrol tracks; actually resetting an account |


In [ ]:
# [CHANGE] each value: prompt / RAG / integration / fine-tune / none
GUESSES = {
    "The circular already exists in an approved handbook.": "RAG",
    "The standing instruction forgot 'never hire'.": "prompt",
    "Which printers are out of warranty? Serials live in a register.": "integration",
    "Plot patrol tracks on a landing site.": "none",
    "Reset this account from the chat box.": "none",
}

gold = {
    "The circular already exists in an approved handbook.": "rag",
    "The standing instruction forgot 'never hire'.": "prompt",
    "Which printers are out of warranty? Serials live in a register.": "integration",
    "Plot patrol tracks on a landing site.": "none",
    "Reset this account from the chat box.": "none",
}
right = 0
for k, g in gold.items():
    got = GUESSES[k].strip().lower()
    ok = got == g
    right += int(ok)
    print(("OK   " if ok else "WRONG"), got, "—", k)
print(f"score {right}/{len(gold)}")
print("Fine-tune is a later, rarer first move. Day 2 builds RAG, not a LoRA.")


## 2. What may enter an **index**

An **index** is the searchable pile. A **chunk** is one paragraph in that pile. **INDEX** = it may be stored for search. **NEVER** = do not copy it in, even “just to try.”

A **canary** is planted sensitive text. If a reply quotes it, the exercise fails.


In [ ]:
# [CHANGE] each value: index or never
GUESSES = {
    "CivicDesk vacancy handbook paragraph C2": "index",
    "A real CV exported from e-recruitment": "never",
    "ICT article: password reset is a human identity check": "index",
    "Cabinet minute body on coastal security": "never",
    "Burgess application with household names": "never",
    "Planted line: National ID 999-00-1111 Maya Reid": "never",
}

gold = {
    "CivicDesk vacancy handbook paragraph C2": "index",
    "A real CV exported from e-recruitment": "never",
    "ICT article: password reset is a human identity check": "index",
    "Cabinet minute body on coastal security": "never",
    "Burgess application with household names": "never",
    "Planted line: National ID 999-00-1111 Maya Reid": "never",
}
right = 0
for k, g in gold.items():
    got = GUESSES[k].strip().lower()
    ok = got == g
    right += int(ok)
    print(("OK   " if ok else "WRONG"), got, "—", k)
print(f"score {right}/{len(gold)}")


## 3. A tiny RAG you can see

The next cell is a **classroom retriever**. It counts shared words. Production systems often use **embeddings** (vectors). Same job: rank chunks. You will not train embeddings tonight.

Four fictional chunks. One is **restricted**.


In [ ]:
CHUNKS = [
    {
        "id": "HB-ATTACH-04",
        "clearance": "internal",
        "text": "Service-request attachments may be stored overnight only on the CivicDesk approved file store. Not personal email. Not a public chatbot.",
    },
    {
        "id": "HB-VAC-C2",
        "clearance": "internal",
        "text": "C2. The applicant must hold a recognised diploma in records management or equivalent. Library studies is not automatically equivalent. Do not hire.",
    },
    {
        "id": "HB-ICT-PWD",
        "clearance": "internal",
        "text": "Password resets are never performed by the chat assistant. Draft a ticket. A human completes identity check.",
    },
    {
        "id": "HB-CAB-X",
        "clearance": "restricted",
        "text": "RESTRICTED. Coastal-security item: fictional harbour-drill watch-window. Clerks must not receive this text.",
    },
]


def words(text):
    out = []
    for w in text.lower().replace(".", " ").replace(",", " ").split():
        if len(w) > 3:
            out.append(w)
    return set(out)


def retrieve(question, k=2):
    q = words(question)
    scored = []
    for c in CHUNKS:
        overlap = q & words(c["id"] + " " + c["text"])
        scored.append({**c, "score": len(overlap), "overlap": sorted(overlap)})
    scored.sort(key=lambda r: -r["score"])
    return [r for r in scored if r["score"] >= 2][:k]


QUESTION = "What is the rule for storing service-request attachments overnight?"
print("QUESTION:", QUESTION)
print()
for h in retrieve(QUESTION):
    print(h["id"], "score", h["score"], "overlap", h["overlap"])
    print(" ", h["text"])
    print()


### Teaching note

`HB-ATTACH-04` should be on top. The **score** is shared words, not permission. **k=2** means “only look at two.” There is no sacred k.


## 4. Cite, invent, or abstain


In [ ]:
def allowed(chunk, role):
    if chunk["clearance"] == "restricted" and role != "records-officer":
        return False
    return True


def answer(question, role="clerk"):
    hits = retrieve(question)
    visible = [h for h in hits if allowed(h, role)]
    if hits and (not visible or visible[0]["id"] != hits[0]["id"]):
        return "abstain", "A classed document matched. This role may not see it. I will not substitute another paragraph. Stop."
    if not visible:
        return "abstain", "I cannot cite. No approved paragraph is in the retrieved set. I will not invent a circular. Stop."
    top = visible[0]
    return "cite", f"Cited {top['id']}. Quoted span: '{top['text']}'"


print("=== prompt-only (no retrieval) — this is Day 1's failure mode ===")
print("CivicDesk Circular 99 of 2024 appears to allow overnight storage anywhere. (Invented. Fail.)")
print()
print("=== RAG on the overnight question ===")
print(answer("What is the rule for storing service-request attachments overnight?"))
print()
print("=== a circular that is not in the pile ===")
print(answer("What does CivicDesk Circular 88 of 2019 say about drone permits?"))


**Hallucination** (extra word): fluent text with no source. Ministries fear that more than a blank. **Grounded** means every factual clause points at a chunk id.


## 5. Access is not retrieval (N4)

The same question, two roles. A **hit in the index** is not permission to display.


In [ ]:
Q = "Show me the CivicDesk coastal-security minute text."
print("QUESTION:", Q)
print("Index hits (before the gate):")
for h in retrieve(Q):
    print(" ", h["id"], h["clearance"], "score", h["score"])
print()
print("clerk         :", answer(Q, role="clerk"))
print("records-officer:", answer(Q, role="records-officer"))


### Teaching note

The clerk should **abstain** even if a weaker public chunk exists — you do not substitute a different paragraph and pretend you answered the Cabinet-shaped question.

Finance EDMS already has access rules. A chatbot does not get a free pass because the UI is friendly.

**Write-up (one sentence in your head):** *If the chunk is not in the retrieved set, or this role may not see it, we will not…*


## Done — Night 2

You should be able to say: **retrieve, then generate; cite or abstain; index vs never; a hit is not permission.**

Last sitting: **PREP-03**. [Open PREP-03 in Colab](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/prep/PREP-03_frozen_and_boundary.ipynb).
